# CS434 PA2 - Models B, C, and D

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import torch

REPO_URL = "https://github.com/syeddaniyalg/cs434-pa2.git"
REPO_DIR = Path("/kaggle/working/cs434-pa2")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print(Path.cwd())

## Hardware

In [ ]:
gpu_count = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {gpu_count}")
for index in range(gpu_count):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
if gpu_count < 2:
    raise RuntimeError("This notebook expects two Kaggle GPUs")

In [ ]:
DATA_DIR = REPO_DIR / "data"
CHECKPOINT_DIR = REPO_DIR / "checkpoints"
ARTIFACT_DIR = REPO_DIR / "artifacts"

def run(command):
    print(" ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True)

def wait_for(processes):
    failures = []
    for label, process in processes:
        return_code = process.wait()
        print(f"{label} finished with code {return_code}", flush=True)
        if return_code != 0:
            failures.append(label)
    if failures:
        raise RuntimeError(f"Failed runs: {failures}")

def train_pair(model):
    processes = []
    for seed, device in ((0, "cuda:0"), (1, "cuda:1")):
        command = [
            sys.executable, "-m", "src.train",
            "--model", model,
            "--seed", seed,
            "--device", device,
            "--data-dir", DATA_DIR,
            "--checkpoint-dir", CHECKPOINT_DIR,
            "--artifact-dir", ARTIFACT_DIR,
            "--workers", 4,
            "--resume",
        ]
        processes.append((f"{model} seed {seed}", subprocess.Popen(list(map(str, command)))))
    wait_for(processes)

def evaluate_pair(models, sym_models=()):
    processes = []
    for seed, device in ((0, "cuda:0"), (1, "cuda:1")):
        command = [
            sys.executable, "-m", "src.evaluate",
            "--models", *models,
            "--sym-models", *sym_models,
            "--seed", seed,
            "--device", device,
            "--data-dir", DATA_DIR,
            "--checkpoint-dir", CHECKPOINT_DIR,
            "--artifact-dir", ARTIFACT_DIR,
            "--workers", 4,
        ]
        command.append("--baselines" if seed == 0 else "--no-baselines")
        processes.append((f"evaluation seed {seed}", subprocess.Popen(list(map(str, command)))))
    wait_for(processes)

## Dataset and exact checks

In [ ]:
run([sys.executable, "-m", "src.data", "--root", DATA_DIR, "--download"])
run([sys.executable, "-m", "src.checks", "--artifact-dir", ARTIFACT_DIR])

## Import model A results

In [ ]:
shutil.copytree("/kaggle/input/datasets/thesyeddaniyal/cs434-pa2-model-a/artifacts", ARTIFACT_DIR, dirs_exist_ok=True)
shutil.copytree("/kaggle/input/datasets/thesyeddaniyal/cs434-pa2-model-a/checkpoints", CHECKPOINT_DIR, dirs_exist_ok=True)
print("/kaggle/input/datasets/thesyeddaniyal/cs434-pa2-model-a")


## Models B, C, and D

In [ ]:
for model in ("B", "C", "D"):
    train_pair(model)

In [ ]:
evaluate_pair(["B", "C", "D"], ["B", "D"])

## Models D+B

In [ ]:
TRAIN_DB = False
FINAL_BASE = "D"

if TRAIN_DB:
    train_pair("DB")
    evaluate_pair(["DB"], ["DB"])
    FINAL_BASE = "DB"

## Final tables, Figure 6, and archive

In [ ]:
run([sys.executable, "-m", "src.results", "--artifact-dir", ARTIFACT_DIR, "--stage", "full", "--final-base", FINAL_BASE])
run([sys.executable, "-m", "src.plotting", "--stage", "final", "--data-dir", DATA_DIR, "--checkpoint-dir", CHECKPOINT_DIR, "--artifact-dir", ARTIFACT_DIR, "--device", "cuda:0", "--final-base", FINAL_BASE])
artifact_archive = shutil.make_archive("/kaggle/working/cs434-pa2-artifacts", "zip", REPO_DIR, base_dir="artifacts")
checkpoint_archive = shutil.make_archive("/kaggle/working/cs434-pa2-checkpoints", "zip", REPO_DIR, base_dir="checkpoints")
print(artifact_archive)
print(checkpoint_archive)